In [1]:
# this magic for develop only
%load_ext autoreload
%autoreload 2

In [2]:
import math
import numpy as np
import asyncio
import xmlschema
import networkx as nx
import polars as pl
from itertools import zip_longest

In [40]:
from ggblab import GeoGebra
ggb = await GeoGebra().init(use_vscode=False)

In [4]:
import ggblab

In [2]:
# import os
# os.environ['GGBLAB_IPYMAGIC_DEBUG'] = '1'

In [3]:
# get_ipython().user_ns['_ggb_debug']=True

In [4]:
# %reload_ext ggblab

In [33]:
cmds = r"""#
# Point(xOyPlane)
(0, 0)
Circle(_1, 1)
Point(_2)
Line(_1, _3)
Point(_4)
PerpendicularLine(_5, _4)
{Intersect(_2, _6)}
_7(1)
_7(2)
Polygon(_1, _8, _9)
# Segment(_1, _8, _10)
# Segment(_8, _9, _10)
# Segment(_9, _1, _10)
Angle(_1, _8, _5)
Angle(_1, _9, _5)
#
Midpoint(_1, _5)
Circle(_, _1)
{Intersect(_2, _)}
_15(1)
_15(2)
Polygon(_1, _5, __)
Angle(_1, ___, _5)
{Tangent(_5, _2)}
_20(1)
_20(2)
"""
for i, c in enumerate([c for c in cmds.splitlines() if not c.startswith('#')], start=1):
    print(f"{i} {c}")

1 (0, 0)
2 Circle(_1, 1)
3 Point(_2)
4 Line(_1, _3)
5 Point(_4)
6 PerpendicularLine(_5, _4)
7 {Intersect(_2, _6)}
8 _7(1)
9 _7(2)
10 Polygon(_1, _8, _9)
11 Angle(_1, _8, _5)
12 Angle(_1, _9, _5)
13 Midpoint(_1, _5)
14 Circle(_, _1)
15 {Intersect(_2, _)}
16 _15(1)
17 _15(2)
18 Polygon(_1, _5, __)
19 Angle(_1, ___, _5)
20 {Tangent(_5, _2)}
21 _20(1)
22 _20(2)


In [31]:
%ggblab api newConstruction()

In [41]:
%ggblab ggb '{cmds}'

ggb.command failed for: 'Intersect(xAxis, b)' -> GeoGebraAppletError: GeoGebra applet error: Command Intersect:
Illegal argument: Line xAxis

Syntax:
Intersect( <Object>, <Object> )
Intersect( <Object>, <Object>, <Index of Intersection Point> )
Intersect( <Object>, <Object>, <Initial Point> )
Intersect( <Function>, <Function>, <Start x-Value>, <End x-Value> )
Intersect( <Curve 1>, <Curve 2>, <Parameter 1>, <Parameter 2> ) [AppletError]


["{'error': 'GeoGebra applet error: Command Intersect:\\nIllegal argument: Line xAxis\\n\\nSyntax:\\nIntersect( <Object>, <Object> )\\nIntersect( <Object>, <Object>, <Index of Intersection Point> )\\nIntersect( <Object>, <Object>, <Initial Point> )\\nIntersect( <Function>, <Function>, <Start x-Value>, <End x-Value> )\\nIntersect( <Curve 1>, <Curve 2>, <Parameter 1>, <Parameter 2> ) [AppletError]'}"]

In [26]:
r1 = await ggb.function('getValueString', ['l1'])
t1 = ggb.parser.tokenize(r1)
t1

['l1', '=', []]

In [27]:
r2 = await ggb.function('getValueString', ['l2'])
t2 = ggb.parser.tokenize(r2)
t2

['l2',
 '=',
 [['0.7646162914027', '0.6444857848871'],
  ['0.7646162914027', '-0.6444857848871']]]

In [28]:
r3 = await ggb.function('getValueString', ['l3'])
t3 = ggb.parser.tokenize(r3, simplify=True)
t3

['l3',
 '=',
 [['-0.6444857848871',
   'x',
   '-',
   '0.5432292400671',
   'y',
   '=',
   '-0.8428878538604'],
  ['0.6444857848871',
   'x',
   '-',
   '0.5432292400671',
   'y',
   '=',
   '0.8428878538604']]]

In [38]:
%ggblab api getCommandString(l1)

'{Intersect(c, g)}'

In [39]:
ggb.parser.tokenize(_)

[['Intersect', [['c', 'g']]]]

In [36]:
from ggblab_extra import ConstructionIO
from ggblab_extra import ConstructionTreeParser

In [90]:
df = await ConstructionIO.initialize_dataframe(ggb, use_applet=True)
df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
u32,str,str,str,str,str,u32,bool,bool,bool
1,"""C_{t}""","""point""","""Point(zAxis)""","""C_{t} = (0, 0, 3)""",null,0,false,true,true
2,"""C_{b}""","""point""","""Point(zAxis)""","""C_{b} = (0, 0, -3)""",null,0,false,true,true
3,"""a""","""cylinder""","""Cylinder(C_{t}, C_{b}, 1)""","""a: 18.85""",null,0,false,false,false
4,"""b""","""surface""","""Cylinder(C_{t}, C_{b}, 1)""","""b: 37.7""",null,0,false,false,false
5,"""c""","""circle""","""Cylinder(C_{t}, C_{b}, 1)""","""c: X = (0, 0, 3) + (cos(t), si…",null,0,false,false,false
6,"""d""","""circle""","""Cylinder(C_{t}, C_{b}, 1)""","""d: X = (0, 0, -3) + (cos(t), -…",null,0,false,false,false
7,"""e""","""circle""","""Circle(yAxis, C_{t})""","""e: X = (0, 0, 0) + (3 sin(t), …",null,3,false,false,false
8,"""Q""","""point""","""Point(e)""","""Q = (2.53, 0, -1.61)""",null,3,true,true,false
9,"""p""","""plane""","""Plane(Q, yAxis)""","""p: -1.61x - 2.53z = 0""",null,3,false,false,false


In [48]:
cmds = ConstructionIO.commands_for_magic(df, use_name_equals=True)
print(cmds)

C_{t} = Point(zAxis)
C_{b} = Point(zAxis)
a = Cylinder(C_{t}, C_{b}, 1)
b = Cylinder(C_{t}, C_{b}, 1)
c = Cylinder(C_{t}, C_{b}, 1)
d = Cylinder(C_{t}, C_{b}, 1)
e = Circle(yAxis, C_{t})
Q = Point(e)
p = Plane(Q, yAxis)
f = IntersectPath(p, a)
q = Plane(xAxis, zAxis)
A = Intersect(q, f)
C = Intersect(q, f)
C' = Intersect(xAxis, b)
A' = Intersect(xAxis, b)
O = Intersect(yAxis, xAxis)
g = Segment(O, A)
i = Segment(O, A')
p_{xy} = Plane(xAxis, yAxis)
k = IntersectPath(p_{xy}, a)
B = Intersect(yAxis, f)
D = Intersect(yAxis, f)
j = Segment(B, O)
c_1 = Circle(yAxis, C)
O'' = Intersect(zAxis, c_1)
O''' = Intersect(zAxis, c_1)
f_2 = PerpendicularLine(O'', p)
f_1 = PerpendicularLine(O''', p)
F_2 = Intersect(f_2, p)
F_1 = Intersect(f_1, p)
g_1 = Segment(B, F_1)
i_1 = Segment(O'', O)
j_1 = Segment(O'', F_2)
k_1 = Segment(F_2, O)
o = Sphere(O'', F_2)
u_{1} = Sphere(O''', F_1)
p_1 = Plane(O'', xOyPlane)
q_1 = Plane(O''', xOyPlane)
d_1 = IntersectPath(p_1, a)
e_1 = IntersectPath(q_1, a)
P = Point(f)

In [86]:
cmds = r"""#
# C_{t} = Point(zAxis)
# C_{b} = Point(zAxis)
C_{t} = (0, 0, 10)
C_{b} = (0, 0, -10)
a = Cylinder(C_{t}, C_{b}, 1)
# b = Cylinder(C_{t}, C_{b}, 1)
# c = Cylinder(C_{t}, C_{b}, 1)
# d = Cylinder(C_{t}, C_{b}, 1)
e = Circle(yAxis, C_{t})
Q = Point(e)
p = Plane(Q, yAxis)
f = IntersectPath(p, a)
q = Plane(xAxis, zAxis)
l1 = {Intersect(q, f)}
A = l1(1)
C = l1(2)
l2 = {Intersect(xAxis, b)}
C' = l2(1)
A' = l2(2)
O = Intersect(yAxis, xAxis)
g = Segment(O, A)
i = Segment(O, A')
p_{xy} = Plane(xAxis, yAxis)
k = IntersectPath(p_{xy}, a)
l3 = {Intersect(yAxis, f)}
B = l3(1)
D = l3(2)
j = Segment(B, O)
c_1 = Circle(yAxis, C)
l4 = {Intersect(zAxis, c_1)}
O'' = l4(1)
O''' = l4(2)
f_2 = PerpendicularLine(O'', p)
f_1 = PerpendicularLine(O''', p)
F_2 = Intersect(f_2, p)
F_1 = Intersect(f_1, p)
g_1 = Segment(B, F_1)
i_1 = Segment(O'', O)
j_1 = Segment(O'', F_2)
k_1 = Segment(F_2, O)
o = Sphere(O'', F_2)
u_{1} = Sphere(O''', F_1)
p_1 = Plane(O'', xOyPlane)
q_1 = Plane(O''', xOyPlane)
d_1 = IntersectPath(p_1, a)
e_1 = IntersectPath(q_1, a)
P = Point(f)
l_1 = Line(P, zAxis)
n_2 = Segment(P, F_2)
n_1 = Segment(P, F_1)
r_1 = Segment(O, F_1)
P'' = Intersect(l_1, d_1)
P''' = Intersect(l_1, e_1)
l = Segment(P'', P)
m = Segment(P, P''')
P_{c} = Intersect(l_1, k)
s = Segment(P, P_{c})
h = Segment(A', A)
h_2 = PerpendicularLine(P_{c}, yAxis)
h_1 = PerpendicularLine(P, yAxis)
P_{h} = Intersect(h_1, h_2)
v_{1} = Distance(P_{h}, P) / Distance(P_{h}, P_{c})
w = Distance(O, A) / Distance(O, A')
axis_{z} = Line(C_{t}, C_{b})
axis_y = Line(O, yAxis)
a_1 = Line(O, Q)
b_1 = Segment(B, F_2)
u = Vector(P_{h}, P)
v = Vector(P_{h}, P_{c})
n = Translate((u v) / (v v) v, P_{h})
h_3 = PerpendicularLine(P_{c}, xAxis, space)
P_{w} = Intersect(h_3, xAxis)
axis_x = Line(O, xAxis)
s_{cw} = Segment(P_{c}, P_{w})
s_{ch} = Segment(P_{c}, P_{h})
i_2 = Segment(O, P_{c})
r = Segment(P, P_{h})
t1 = Polygon(O, P_{c}, P_{h})
p_{h} = Segment(O, P_{c}, t1)
o_1 = Segment(P_{c}, P_{h}, t1)
p_{c} = Segment(P_{h}, O, t1)
m_1 = Distance(O, P_{h})² + Distance(O, P_{w})²
t = Line(O, P_{w})
t_1 = Segment(O, P_{h})
j_2 = Segment(P_{h}, P_{c})
l4 = {Intersect(axis_y, k)}
E = l4(1)
F = l4(2)
s_1 = Segment(E, F)
g_2 = Segment(O, P)
t2 = Polygon(O, P, P_{h})
k_2 = Segment(O, P, t2)
o_2 = Segment(P, P_{h}, t2)
p_2 = Segment(P_{h}, O, t2)
"""
for i, c in enumerate([c for c in cmds.splitlines() if not c.startswith('#')], start=1):
    print(f"{i} {c}")

1 C_{t} = (0, 0, 10)
2 C_{b} = (0, 0, -10)
3 a = Cylinder(C_{t}, C_{b}, 1)
4 e = Circle(yAxis, C_{t})
5 Q = Point(e)
6 p = Plane(Q, yAxis)
7 f = IntersectPath(p, a)
8 q = Plane(xAxis, zAxis)
9 l1 = {Intersect(q, f)}
10 A = l1(1)
11 C = l1(2)
12 l2 = {Intersect(xAxis, b)}
13 C' = l2(1)
14 A' = l2(2)
15 O = Intersect(yAxis, xAxis)
16 g = Segment(O, A)
17 i = Segment(O, A')
18 p_{xy} = Plane(xAxis, yAxis)
19 k = IntersectPath(p_{xy}, a)
20 l3 = {Intersect(yAxis, f)}
21 B = l3(1)
22 D = l3(2)
23 j = Segment(B, O)
24 c_1 = Circle(yAxis, C)
25 l4 = {Intersect(zAxis, c_1)}
26 O'' = l4(1)
27 O''' = l4(2)
28 f_2 = PerpendicularLine(O'', p)
29 f_1 = PerpendicularLine(O''', p)
30 F_2 = Intersect(f_2, p)
31 F_1 = Intersect(f_1, p)
32 g_1 = Segment(B, F_1)
33 i_1 = Segment(O'', O)
34 j_1 = Segment(O'', F_2)
35 k_1 = Segment(F_2, O)
36 o = Sphere(O'', F_2)
37 u_{1} = Sphere(O''', F_1)
38 p_1 = Plane(O'', xOyPlane)
39 q_1 = Plane(O''', xOyPlane)
40 d_1 = IntersectPath(p_1, a)
41 e_1 = IntersectPath(q

In [82]:
%ggblab api newConstruction()

In [87]:
%ggblab ggb "{cmds}"

['C_{t}',
 'C_{b}',
 'a',
 'e',
 'Q',
 'p',
 'f',
 'q',
 'l1',
 'A',
 'C',
 'l2',
 "C'",
 "A'",
 'O',
 'g',
 'i',
 'p_{xy}',
 'k',
 'l3',
 'B',
 'D',
 'j',
 'c_1',
 'l4',
 "O''",
 "O'''",
 'f_2',
 'f_1',
 'F_2',
 'F_1',
 'g_1',
 'i_1',
 'j_1',
 'k_1',
 'o',
 'u_{1}',
 'p_1',
 'q_1',
 'd_1',
 'e_1',
 'P',
 'l_1',
 'n_2',
 'n_1',
 'r_1',
 "P''",
 "P'''",
 'l',
 'm',
 'P_{c}',
 's',
 'h',
 'h_2',
 'h_1',
 'P_{h}',
 'v_{1}',
 'w',
 'axis_{z}',
 'axis_y',
 'a_1',
 'b_1',
 'u',
 'v',
 'n',
 'h_{3}',
 'P_{w}',
 'axis_x',
 's_{cw}',
 's_{ch}',
 'i_2',
 'r',
 't1',
 'None',
 'None',
 'None',
 'm_1',
 't',
 't_1',
 'j_2',
 'l4',
 'E',
 'F',
 's_1',
 'g_2',
 't2',
 'None',
 'None',
 'None']

In [4]:
%pwd

'/Users/manabu/work/ggblab/examples'

In [7]:
%cd ggblab/examples

/Users/manabu/work/ggblab/examples


In [88]:
# ggb.file.load('eg11_slider.ggb')
ggb.file.load('2025_02_03.ggb')

In [89]:
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

In [51]:
from ggblab.errors import GeoGebraError, GeoGebraSyntaxError, GeoGebraSemanticsError, GeoGebraAppletError

In [56]:
ggb.check_semantics = False

In [162]:
from ggblab_extra import ConstructionTreeParser

In [163]:
p = ConstructionTreeParser(df, auto_assign_layers=True)

In [176]:
g1 = p.parse()

In [177]:
p.df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary,DependsOn
u32,str,str,str,str,str,i64,bool,bool,bool,list[str]
1,"""A""","""point""","""(0, 0)""","""A = (0, 0)""",null,0,false,false,false,[]
2,"""c""","""circle""","""Circle(A, 1)""","""c: x² + y² = 1""",null,0,true,true,false,"[""A""]"
3,"""B""","""point""","""Point(c)""","""B = (1, 0)""",null,0,true,true,false,"[""A"", ""c""]"
4,"""f""","""line""","""Line(A, B)""","""f: y = 0""",null,0,true,true,false,"[""A"", ""B"", ""c""]"
5,"""C""","""point""","""Point(f)""","""C = (0, 0)""",null,0,true,true,false,"[""A"", ""B"", … ""f""]"
6,"""g""","""line""","""PerpendicularLine(C, f)""","""g: x = 0""",null,0,true,true,false,"[""A"", ""B"", … ""f""]"
7,"""l1""","""list""","""{Intersect(c, g)}""","""l1 = {(0, -1), (0, 1)}""",null,0,true,true,false,[]
8,"""D""","""point""","""l1(1)""","""D = (0, -1)""",null,0,true,true,false,"[""l1""]"
9,"""E""","""point""","""l1(2)""","""E = (0, 1)""",null,0,true,true,false,"[""l1""]"


In [168]:
p.df["Name", "Type", "Command", "Layer"]

Name,Type,Command,Layer
str,str,str,i64
"""A""","""point""","""(0, 0)""",6
"""c""","""circle""","""Circle(A, 1)""",6
"""B""","""point""","""Point(c)""",1
"""f""","""line""","""Line(A, B)""",2
"""C""","""point""","""Point(f)""",9
"""g""","""line""","""PerpendicularLine(C, f)""",7
"""l1""","""list""","""{Intersect(c, g)}""",8
"""D""","""point""","""l1(1)""",3
"""E""","""point""","""l1(2)""",3


In [148]:
_oh

{6: ['A',
  'c',
  'B',
  'f',
  'C',
  'g',
  'l1',
  'D',
  'E',
  't1',
  'F',
  'h',
  'l2',
  'G',
  'H',
  't2',
  'l3',
  'undefined',
  'undefined'],
 8: shape: (25, 10)
 ┌──────────┬──────┬──────────┬────────────────────┬───┬───────┬────────────┬───────────┬───────────┐
 │ Sequence ┆ Name ┆ Type     ┆ Command            ┆ … ┆ Layer ┆ ShowObject ┆ ShowLabel ┆ Auxiliary │
 │ ---      ┆ ---  ┆ ---      ┆ ---                ┆   ┆ ---   ┆ ---        ┆ ---       ┆ ---       │
 │ u32      ┆ str  ┆ str      ┆ str                ┆   ┆ u32   ┆ bool       ┆ bool      ┆ bool      │
 ╞══════════╪══════╪══════════╪════════════════════╪═══╪═══════╪════════════╪═══════════╪═══════════╡
 │ 1        ┆ A    ┆ point    ┆ (0, 0)             ┆ … ┆ 0     ┆ false      ┆ false     ┆ false     │
 │ 2        ┆ c    ┆ circle   ┆ Circle(A, 1)       ┆ … ┆ 0     ┆ true       ┆ true      ┆ false     │
 │ 3        ┆ B    ┆ point    ┆ Point(c)           ┆ … ┆ 0     ┆ true       ┆ true      ┆ false     │
 │ 4  

In [169]:
await ggb.function("setLayer", list(zip(*p.df["Name", "Layer"])))

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [178]:
labels_map = {}
for n, t in p.df["Name", "Type"].rows():
    try:
        g1.nodes[n]['label'] = f"{n} ({t})"
    except:
        pass

In [179]:
nx.set_node_attributes(g1, labels_map, "label")
nx.write_network_text(g1, with_labels="label")

╟── A (point)
╎   ├─╼ c (circle)
╎   │   ├─╼ B (point)
╎   │   │   └─╼ f (line) ╾ A (point)
╎   │   │       ├─╼ C (point)
╎   │   │       │   ├─╼ g (line) ╾ f (line)
╎   │   │       │   │   └─╼ l1__list ╾ c (circle)
╎   │   │       │   │       ├─╼ D (point) ╾ l1 (list)
╎   │   │       │   │       │   ├─╼ t1 (triangle) ╾ A (point), E (point)
╎   │   │       │   │       │   │   ├─╼ e (segment) ╾ A (point), D (point)
╎   │   │       │   │       │   │   ├─╼ a (segment) ╾ D (point), E (point)
╎   │   │       │   │       │   │   └─╼ d (segment) ╾ E (point), A (point)
╎   │   │       │   │       │   └─╼  ...
╎   │   │       │   │       └─╼ E (point) ╾ l1 (list)
╎   │   │       │   │           └─╼  ...
╎   │   │       │   ├─╼ F (point) ╾ A (point)
╎   │   │       │   │   └─╼ h (point) ╾ A (point)
╎   │   │       │   │       └─╼ l2__list ╾ c (circle)
╎   │   │       │   │           ├─╼ G (point) ╾ l2 (list)
╎   │   │       │   │           │   └─╼ t2 (triangle) ╾ A (point), C (point)
╎   │   │  

In [43]:
g2 = p.parse_subgraph()
nx.write_network_text(g2)

╙── A
    └─╼ c
        ├─╼ B
        │   ├─╼ C ╾ f
        │   │   ├─╼ h
        │   │   │   ├─╼ G ╾ l2__list
        │   │   │   └─╼ H ╾ l2__list
        │   │   ├─╼ l1__list
        │   │   │   ├─╼ E
        │   │   │   └─╼ D
        │   │   ├─╼ j ╾ l3__list
        │   │   ├─╼ i ╾ l3__list
        │   │   └─╼ l2__list ╾ F
        │   │       └─╼  ...
        │   ├─╼ l3__list ╾ f
        │   │   └─╼  ...
        │   ├─╼ g ╾ f
        │   └─╼ F ╾ f
        │       └─╼  ...
        └─╼ f
            └─╼  ...


In [44]:
from ggblab_extra import hungarian_similarity

In [45]:
s, r = hungarian_similarity(g1, g2)
s

np.float64(0.8571428571428572)

In [33]:
p.df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary,DependsOn,DependsOn_minimal
u32,str,str,str,str,str,u32,bool,bool,bool,list[str],list[str]
0,"""A""","""point""","""(0, 0)""","""A = (0, 0)""",null,0,false,false,false,[],[]
1,"""c""","""circle""","""Circle(A, 1)""","""c: x² + y² = 1""",null,0,true,true,false,"[""A""]","[""A""]"
2,"""B""","""point""","""Point(c)""","""B = (-1, 0)""",null,0,false,false,false,"[""A"", ""c""]","[""c""]"
3,"""f""","""line""","""Line(A, B)""","""f: y = 0""",null,0,false,false,false,"[""A"", ""B"", ""c""]","[""c""]"
4,"""C""","""point""","""Point(f)""","""C = (1, 0)""",null,0,false,false,false,"[""A"", ""B"", … ""f""]","[""B"", ""f""]"
5,"""g""","""line""","""PerpendicularLine(C, f)""","""g: x = 1""",null,0,false,false,false,"[""A"", ""B"", … ""f""]","[""B"", ""f""]"
6,"""l1""","""list""","""{Intersect(c, g)}""","""l1 = {(1, 0)}""",null,0,true,true,false,[],[]
7,"""D""","""point""","""l1(1)""","""D = (1, 0)""",null,0,false,false,false,"[""l1""]","[""l1""]"
8,"""E""","""point""","""l1(2)""","""E = (?, ?)""",null,0,true,true,false,"[""l1""]","[""l1""]"


In [12]:
l1 = 'n'
l2 = 'i'

In [13]:
await ggb.listen(l1, True)
await ggb.listen(l2, True)

{}

In [141]:
await ggb.listen('a', False)
await ggb.listen('b', False)

{}

In [14]:
ggb.comm.shared_objects

{'n': 'n = 0', 'i': 'i = 0'}

In [15]:
# import ipywidgets as widgets
# label1 = widgets.Label(value=ggb.comm.shared_objects['n'])
# label2 = widgets.Label(value=ggb.comm.shared_objects['m'])
# display(label1, label2)

In [16]:
async def on_shared_update1(changes):
    # await asyncio.sleep(0)
    # label1.value = changes['a']
    n = int(changes[l1].split()[2])
    # await asyncio.sleep(0)
    await ggb.function("setLayerVisible", list(zip_longest(range(9), [True]*n, fillvalue=False)))
    r = await ggb.function('getXML', [l2])
    o = ggb.file.ggb_schema.decode(r)
    o['value'][0]['@val'] = '0'
    x = xmlschema.etree_tostring(ggb.file.ggb_schema.encode(o, 'element'))
    r = await ggb.function('evalXML' , [x])

In [17]:
ggb.comm.remove_shared_listener(on_shared_update1)
ggb.comm.add_shared_listener(on_shared_update1)

True

In [18]:
async def on_shared_update2(changes):
    # await asyncio.sleep(0)
    # label2.value = changes['b']
    m = int(changes[l2].split()[2])
    n = int(ggb.comm.shared_objects[l1].split()[2])
    l = df.filter(pl.col("Layer") == n)["Name"].to_list()
    # m = int(ggb.comm.shared_objects['m'].split()[2])
    # list(zip_longest(l, [True]*m, fillvalue=False))
    await ggb.function("setVisible", list(zip_longest(l, [True]*m, fillvalue=False)))

In [19]:
ggb.comm.remove_shared_listener(on_shared_update2)
ggb.comm.add_shared_listener(on_shared_update2)

True

In [104]:
ggb.comm.clear_shared_listeners()

2

In [29]:
await ggb.command('Curve(x, x^2, x, -10, 10)')

'a'

In [30]:
await ggb.command('a(b)')

'G'

In [32]:
await ggb.command('Tangent(G, a)')

'h'

In [33]:
await ggb.command('u w')

'e'

In [34]:
await ggb.command('sqrt(u u)')

'j'

In [36]:
await ggb.command('k = Distance(a,F)')

'k'

In [7]:
await ggb.command('{Tangent(c,C)}')

'l2'

In [17]:
r1 = await ggb.function('getValueString', ['l1'])
# l1 = ggb.parser.tokenize_with_commas(r1)
l2 = ggb.parser.tokenize(r1)
l2

['l1',
 '=',
 [['0.6052896362051', '-0.7960053117302'],
  ['0.6052896362051', '0.7960053117302']]]

In [18]:
r2 = await ggb.function('getValueString', ['l2'])
l = ggb.parser.tokenize(r2, simplify=True)
l

[]

In [52]:
await ggb.command("l1(2)")

'E'

In [50]:
await ggb.command("{Intersect[c, g]}")

'l1'

In [200]:
async def getCoords(list_points=[]):
    r = await ggb.function(["getXcoord", "getYcoord"], [[p] for p in list_points])
    arr = np.array(list(zip(*r)), dtype=float)
    arr[np.isclose(arr, 0., atol=1e-9)] = 0.
    arr = arr[~np.any(np.isnan(arr), axis=1)]
    return arr.tolist()

In [219]:
await getCoords(['D', 'E'])

[[0.8000000000000002, 0.5999999999999999], [0.7999999999999999, -0.6]]

In [222]:
r = await ggb.function("getValueString", ['l1'])
len(list(toCoords(r)))

2

In [215]:
from collections.abc import Iterable

def toCoords(r):
    # ret = []
    for e in ggb.parser.tokenize_with_commas(r):
        if isinstance(e, Iterable) and not isinstance(e, (str, bytes)):
            r2 = [float(e2) for e2 in e if e2 not in [',', '?']]
            if r2:
                # ret.append(r2)
                yield r2
    # return ret


In [223]:
import ipywidgets as widgets
label = widgets.Label(value="")
display(label)

Label(value='')

In [224]:
await ggb.listen('l1')

{}

In [225]:
ggb.comm.shared_objects

{'l1': 'l1 = {(0.8, 0.6), (0.8, -0.6)}'}

In [226]:
async def on_shared_update(changes):
    n = len(list(toCoords(changes['l1'])))
    label.value = f"Length of l1: {n}"

In [227]:
ggb.comm.add_shared_listener(on_shared_update)

True

In [6]:
%pwd

'/Users/manabu/work/ggblab'

In [ ]:
# ggb.file.source_file = 'eg11_slider.ggb'

In [39]:
ggb.file.base64_buffer = await ggb.function("getBase64")

In [40]:
ggb.file.save(overwrite=True)

* 原則（教育観）:
    - 目的化: 再現は「結果」ではなく「理解（なぜその操作か）」を目的にする。
    - 予測→検証: 次に何が起きるか予測させてから操作させる。
    - 説明要求: 手順ごとに短い理由説明（1文）を書かせる。
    - 変奏課題: パラメータを少し変えた課題で本質が移るか確認する。
    - 生成的課題: 「同じ発想で別の図形を作る」など転移を問う。
* ggblabで実装できる仕組み（短）:
    - 段階公開（layer slider）: 各レイヤーに「解説」「問い」「期待する操作」を紐付け、スライダーで段階的に提示。
    - 予測プロンプト: 各ステップの前に「次に何が起きる？」を表示し、回答を記録。
    - 説明入力欄: 学生が操作毎に短い説明を入力 → 教師や自動ルールでフィードバック。
    - 変化タスク自動化: DataFrame→コマンド生成を利用してパラメータをランダム化した派生課題を作る。
    - 操作ログ＋解析: 操作順・所要時間・試行回数をログ化して学習診断に使う。
    - 差分フィードバック: 学生構成と模範構成を比較して「次に直すべき一手」を提示。

In [29]:
await ggb.function("getVersion")

'5.2.909.9'